## Week 4 Exercise

# The Sidekick

### Package Imports

In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv

### Loading the Environment Var

In [ ]:
load_dotenv(override=True)

### Definition of Structured Output with Pydantic Object

In [ ]:
# First define structured outputs

class ClarificationOutput(BaseModel):
    needs_clarification: bool = Field(
        description="True if the request or success criteria are too ambiguous to hand off to the worker without asking the user something specific first"
    )
    question: Optional[str] = Field(
        default=None,
        description="If clarification is needed, a clear question for the user (plain text, no prefix required)",
    )


class PlannerOutput(BaseModel):
    steps: List[str] = Field(
        description="Ordered steps for the execution agent to follow to meet the success criteria"
    )
    notes: Optional[str] = Field(
        default=None,
        description="Optional assumptions, constraints, or risks the worker should keep in mind",
    )


class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if more input is needed from the user, or clarifications, or the assistant is stuck")


### Definition of TypedDict

In [ ]:
# The state

class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool
    awaiting_user_clarification: bool
    plan: Optional[str]

### Get Async Playwright tools

In [ ]:

import nest_asyncio
nest_asyncio.apply()
async_browser =  create_async_playwright_browser(headless=False)  # headful mode
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

### Initialize the LLMs

In [ ]:
clarifier_llm = ChatOpenAI(model="gpt-4o-mini")
clarifier_llm_with_output = clarifier_llm.with_structured_output(ClarificationOutput)

planner_llm = ChatOpenAI(model="gpt-4o-mini")
planner_llm_with_output = planner_llm.with_structured_output(PlannerOutput)

worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

### Utility Functions & Logic

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    """Turn the message list into a simple readable transcript for prompts (user vs assistant lines)."""
    conversation = "Conversation history:\n\n"
    for message in messages:
        if isinstance(message, HumanMessage):
            conversation += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            text = message.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation


def clarification(state: State) -> Dict[str, Any]:
    """Ask the user a focused question if the task is too vague; otherwise let the flow continue to planning."""
    system_message = """You are a clarification specialist. Your only job is to decide whether the user's request
together with the success criteria is specific enough for a task-execution agent to proceed.
If critical details are missing or contradictory, you must ask one focused question.
If enough is clear to start work (even if some details could be refined later), do not ask for clarification."""
    user_message = f"""Success criteria for this assignment:
{state['success_criteria']}

Conversation so far:
{format_conversation(state['messages'])}

Decide whether you need to ask the user a clarifying question before handing off to the worker agent."""
    out = clarifier_llm_with_output.invoke(
        [SystemMessage(content=system_message), HumanMessage(content=user_message)]
    )
    if out.needs_clarification:
        text = (out.question or "").strip() or "Could you add a bit more detail so I can proceed?"
        return {
            "messages": [AIMessage(content=text)],
            "awaiting_user_clarification": True,
        }
    return {"awaiting_user_clarification": False}


def clarification_router(state: State) -> str:
    """Stop after a clarification question, or send the run onward to the planner."""

    if state.get("awaiting_user_clarification"):
        return "END"
    return "planner"


def planner(state: State) -> Dict[str, Any]:
    """Build a short numbered plan from the conversation and success criteria."""

    system_message = """You are a planning specialist. Given the user's messages and success criteria, produce a concise
ordered plan for an execution agent that can use browser/tools. Do not execute tasks yourself—only outline steps.
Prefer specific, actionable steps; keep the list short enough to follow in one session."""
    user_message = f"""Success criteria:
{state['success_criteria']}

Conversation:
{format_conversation(state['messages'])}

Produce a step-by-step plan."""
    out = planner_llm_with_output.invoke(
        [SystemMessage(content=system_message), HumanMessage(content=user_message)]
    )
    steps = out.steps or ["Address the request using tools as needed to meet the success criteria."]
    plan_text = "\n".join(f"{i + 1}. {s}" for i, s in enumerate(steps))
    if out.notes:
        plan_text += f"\n\nNotes: {out.notes}"
    return {"plan": plan_text}


def worker(state: State) -> Dict[str, Any]:
    """Run the tool-using assistant: follow the plan and success criteria, then reply or call tools."""
    
    plan_block = state.get("plan") or ""
    plan_section = (
        f"\n\nPlan to follow (adapt if the situation changes):\n{plan_block}\n"
        if plan_block
        else ""
    )
    system_message = f"""You are a helpful assistant that can use tools to complete tasks.
Work toward satisfying the success criteria below. Use tools when needed, reason step by step, and when you are done
respond with your final answer in natural language (no special prefix required).
{plan_section}
Success criteria:
{state['success_criteria']}
"""
    if state.get("feedback_on_work"):
        system_message += f"""
Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
Here is the feedback on why this was rejected:
{state['feedback_on_work']}
With this feedback, continue and meet the success criteria."""
    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages
    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}

### Worker Definition

In [ ]:
def worker_router(state: State) -> str:
    """After the worker speaks: run Playwright tools next, or go to evaluation if there is nothing to execute."""
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "evaluator"

### Result Evaluator

In [ ]:
def evaluator(state: State) -> State:
    """Judge the worker's latest answer: met criteria, need user input, or retry with feedback in state."""
    last_response = state["messages"][-1].content

    system_message = """You are an evaluator that determines if a task has been completed successfully by an Assistant.
Assess the Assistant's last response based on the given criteria. Respond with your feedback, and with your decision on whether the success criteria has been met,
and whether more input is needed from the user."""
    
    user_message = f"""You are evaluating a conversation between the User and Assistant. You decide what action to take based on the last response from the Assistant.

The entire conversation with the assistant, with the user's original request and all replies, is:
{format_conversation(state['messages'])}

The success criteria for this assignment is:
{state['success_criteria']}

And the final response from the Assistant that you are evaluating is:
{last_response}

Respond with your feedback, and decide if the success criteria is met by this response.
Also, decide if more user input is required, either because the assistant has a question, needs clarification, or seems to be stuck and unable to answer without help.
"""
    if state["feedback_on_work"]:
        user_message += f"Also, note that in a prior attempt from the Assistant, you provided this feedback: {state['feedback_on_work']}\n"
        user_message += "If you're seeing the Assistant repeating the same mistakes, then consider responding that user input is required."
    
    evaluator_messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

    eval_result = evaluator_llm_with_output.invoke(evaluator_messages)
    new_state = {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback on this answer: {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed
    }
    return new_state

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    """Finish the graph if we're done or need the user; otherwise send another pass back to the worker."""
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    else:
        return "worker"

### Graph Builder

In [ ]:
# Set up Graph Builder with State
graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("clarification", clarification)
graph_builder.add_node("planner", planner)
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_node("evaluator", evaluator)

# Add edges
graph_builder.add_edge(START, "clarification")
graph_builder.add_conditional_edges(
    "clarification", clarification_router, {"planner": "planner", "END": END}
)
graph_builder.add_edge("planner", "worker")
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})

# Compile the graph
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

### Gradio UI Setup

In [ ]:
def make_thread_id() -> str:
    """Fresh id for LangGraph checkpointing so each chat thread keeps its own state."""
    return str(uuid.uuid4())


async def process_message(message, success_criteria, history, thread):
    """Run one graph invocation and append the right chat messages (clarification-only vs worker + evaluator)."""
    config = {"configurable": {"thread_id": thread}}

    state = {
        "messages": message,
        "success_criteria": success_criteria,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
        "awaiting_user_clarification": False,
        "plan": None,
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    if result.get("awaiting_user_clarification"):
        clarification_reply = {"role": "assistant", "content": result["messages"][-1].content}
        return history + [user, clarification_reply]
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]

async def reset():
    """Clear inputs, chat, and start a new thread id for a clean conversation."""
    return "", "", None, make_thread_id()



### And now launch our Sidekick UI

In [ ]:

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Sidekick Personal Co-worker")
    thread = gr.State(make_thread_id())
    
    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to your sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(show_label=False, placeholder="What are your success critiera?")
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")
    message.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    success_criteria.submit(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    go_button.click(process_message, [message, success_criteria, chatbot, thread], [chatbot])
    reset_button.click(reset, [], [message, success_criteria, chatbot, thread])

    
demo.launch()